# 03 - Prompt Generation

## Amaç

Bu notebook, CHRD veri setinde seçilen görseller için otomatik olarak metin istemleri (prompt) üretmek amacıyla hazırlanmıştır.

Bu aşamada;

- metadata_selected.csv dosyası okunacaktır.
- Her görsel için Türkçe prompt oluşturulacaktır.
- Aynı prompt İngilizceye çevrilecektir.
- Oluşturulan promptlar yeni bir CSV dosyasına kaydedilecektir.

Bu promptlar daha sonraki aşamada Stable Diffusion XL, FLUX ve DALL-E modellerine gönderilecektir.

## Gerekli Kütüphaneler

Bu bölümde veri işlemek için gerekli Python kütüphaneleri yüklenmektedir.

In [30]:
import pandas as pd
import os

print("Pandas:", pd.__version__)

Pandas: 3.0.3


## Metadata Dosyasını Okuma

Bir önceki aşamada hazırlanan metadata_selected.csv dosyası okunmaktadır.

In [18]:
df = pd.read_csv("../data/CHRD_Real/metadata_selected.csv", sep=";")
df.head()

,image_id,file_name,theme,location,motif_1,motif_2,season,contains_people,quality_score,cultural_value,notes
0,CHRD_REAL_0001,CHRD_REAL_0001.jpeg,Rock-cut Heritage,Ürgüp,Cave House,Stone House,Summer,No,NaN,High,NaN
1,CHRD_REAL_0004,CHRD_REAL_0004.jpeg,Village Square,Ürgüp,Street,Local People,Summer,No,NaN,Medium,NaN
2,CHRD_REAL_0005,CHRD_REAL_0005.jpeg,Fairy Chimney,NaN,Rock Formation,Balloon,Summer,No,NaN,High,NaN
3,CHRD_REAL_0006,CHRD_REAL_0006.jpeg,Local Architecture,NaN,Church,Monastery,Summer,No,NaN,Low,NaN
4,CHRD_REAL_0007,CHRD_REAL_0007.jpeg,Daily Life,NaN,Stone House,Street,Summer,No,NaN,Medium,NaN


## Veri Seti Boyutu

Kaç adet görsel ile çalışacağımız kontrol edilir.

In [19]:
print("Toplam Görsel:", len(df))

Toplam Görsel: 164


## Kullanılacak Alanların Kontrolü

Prompt üretiminde kullanılacak sütunlar incelenmektedir.

In [20]:
print(df.columns)

Index(['image_id', 'file_name', 'theme', 'location', 'motif_1', 'motif_2',
       'season', 'contains_people', 'quality_score', 'cultural_value',
       'notes'],
      dtype='str')


## Türkçe Prompt Oluşturma

Bu bölümde metadata bilgilerinden doğal Türkçe açıklamalar oluşturulmaktadır.

In [21]:
def create_prompt_tr(row):

    prompt = f"{row['theme']} temalı"

    if pd.notna(row["location"]):
        prompt += f", {row['location']} bölgesinde"

    if pd.notna(row["motif_1"]):
        prompt += f", {row['motif_1']}"

    if pd.notna(row["motif_2"]):
        prompt += f" ve {row['motif_2']}"

    prompt += " içeren gerçekçi Kapadokya fotoğrafı."

    return prompt

In [22]:
df["prompt_tr"] = df.apply(create_prompt_tr, axis=1)

In [23]:
df[["theme","prompt_tr"]].head()

,theme,prompt_tr
0,Rock-cut Heritage,"Rock-cut Heritage temalı, Ürgüp bölgesinde, Ca..."
1,Village Square,"Village Square temalı, Ürgüp bölgesinde, Stree..."
2,Fairy Chimney,"Fairy Chimney temalı, Rock Formation ve Balloo..."
3,Local Architecture,"Local Architecture temalı, Church ve Monastery..."
4,Daily Life,"Daily Life temalı, Stone House ve Street içere..."


## İngilizce Prompt Oluşturma

Türkçe prompt yapısının İngilizce karşılığı oluşturulmaktadır.

In [24]:
def create_prompt_en(row):

    prompt = f"A realistic photograph of {row['theme']}"

    if pd.notna(row["location"]):
        prompt += f" in {row['location']}"

    if pd.notna(row["motif_1"]):
        prompt += f", featuring {row['motif_1']}"

    if pd.notna(row["motif_2"]):
        prompt += f" and {row['motif_2']}"

    prompt += ", Cappadocia, Turkey."

    return prompt

In [25]:
df["prompt_en"] = df.apply(create_prompt_en, axis=1)

In [26]:
df[["theme","prompt_en"]].head()

,theme,prompt_en
0,Rock-cut Heritage,A realistic photograph of Rock-cut Heritage in...
1,Village Square,A realistic photograph of Village Square in Ür...
2,Fairy Chimney,"A realistic photograph of Fairy Chimney, featu..."
3,Local Architecture,"A realistic photograph of Local Architecture, ..."
4,Daily Life,"A realistic photograph of Daily Life, featurin..."


## Oluşturulan Promptların İncelenmesi

Türkçe ve İngilizce promptlar birlikte görüntülenmektedir.

In [27]:
df[["theme","prompt_tr","prompt_en"]].head(10)

,theme,prompt_tr,prompt_en
0,Rock-cut Heritage,"Rock-cut Heritage temalı, Ürgüp bölgesinde, Ca...",A realistic photograph of Rock-cut Heritage in...
1,Village Square,"Village Square temalı, Ürgüp bölgesinde, Stree...",A realistic photograph of Village Square in Ür...
2,Fairy Chimney,"Fairy Chimney temalı, Rock Formation ve Balloo...","A realistic photograph of Fairy Chimney, featu..."
3,Local Architecture,"Local Architecture temalı, Church ve Monastery...","A realistic photograph of Local Architecture, ..."
4,Daily Life,"Daily Life temalı, Stone House ve Street içere...","A realistic photograph of Daily Life, featurin..."
5,Local Architecture,"Local Architecture temalı, Rock Formation ve S...","A realistic photograph of Local Architecture, ..."
6,Daily Life,"Daily Life temalı, Stone House ve Pottery içer...","A realistic photograph of Daily Life, featurin..."
7,Natural Landscape,"Natural Landscape temalı, Sunrise ve Balloon i...","A realistic photograph of Natural Landscape, f..."
8,Daily Life,"Daily Life temalı, Valley ve Rock Formation iç...","A realistic photograph of Daily Life, featurin..."
9,Village Square,"Village Square temalı, Rock Formation ve Lands...","A realistic photograph of Village Square, feat..."


## Prompt Dosyasını Kaydetme

Oluşturulan promptlar daha sonraki aşamalarda kullanılmak üzere CSV dosyasına kaydedilmektedir.

In [28]:
df.to_csv(
    "../data/CHRD_Real/prompts.csv",
    sep=";",
    index=False
)

## Kontrol

Prompt dosyasının başarıyla oluşturulduğu doğrulanmaktadır.

In [29]:
print("Prompt dosyası oluşturuldu.")

print()

print(df.shape)

print()

print(df[["prompt_tr","prompt_en"]].head())

Prompt dosyası oluşturuldu.

(164, 13)

                                           prompt_tr  \
0  Rock-cut Heritage temalı, Ürgüp bölgesinde, Ca...   
1  Village Square temalı, Ürgüp bölgesinde, Stree...   
2  Fairy Chimney temalı, Rock Formation ve Balloo...   
3  Local Architecture temalı, Church ve Monastery...   
4  Daily Life temalı, Stone House ve Street içere...   

                                           prompt_en  
0  A realistic photograph of Rock-cut Heritage in...  
1  A realistic photograph of Village Square in Ür...  
2  A realistic photograph of Fairy Chimney, featu...  
3  A realistic photograph of Local Architecture, ...  
4  A realistic photograph of Daily Life, featurin...  
